# NLP Analysis — Consumer Complaint Narratives

This notebook explores the free-text complaint narratives: word/n-gram frequency, topic modeling (LDA), and baseline classifiers for predicting `Product` and `Issue` from the narrative text. The final cleaned-text + tuned-classifier approach here is what `nlp_model_training.py` runs on the full dataset.

**Note:** All exploration in this notebook runs on a 50,000-row sample for speed. The production training script samples up to 300,000 rows.

In [ ]:
import re

import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

## 1. Load Data & Sample

In [ ]:
df = pd.read_parquet(
    "../data/processed/complaints_processed.parquet",
    columns=[
        "Consumer complaint narrative",
        "Company",
        "Product",
        "Issue"
    ]
)

In [ ]:
df["Consumer complaint narrative"].isna().mean() * 100

In [ ]:
sample_df = (
    df[df["Consumer complaint narrative"].notna()]
    .sample(n=50_000, random_state=42)
    .copy()
)

sample_df.shape

In [ ]:
sample_df.head(2)

## 2. Narrative Length & Category Distribution

In [ ]:
sample_df["word_count"] = (
    sample_df["Consumer complaint narrative"]
    .str.split()
    .str.len()
)

sample_df["word_count"].describe()

In [ ]:
sample_df["Product"].value_counts().head(20)

In [ ]:
sample_df["Issue"].value_counts().head(20)

In [ ]:
product_length = (
    sample_df.groupby("Product")["word_count"]
    .mean()
    .sort_values(ascending=False)
)

product_length

In [ ]:
issue_length = (
    sample_df.groupby("Issue")["word_count"]
    .mean()
    .sort_values(ascending=False)
)

issue_length.head(10)

## 3. Text Cleaning

Lowercase, strip CFPB's masked tokens (`XXXX`), remove digits and punctuation, collapse whitespace. This mirrors `clean_text()` in `nlp_predictor.py` / `nlp_model_training.py`, so word/n-gram frequencies below reflect what the production model actually sees.

In [ ]:
def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"\bxx+\b", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


sample_df["clean_text"] = sample_df["Consumer complaint narrative"].apply(clean_text)
sample_df.head(3)

## 4. Word & N-Gram Frequency (Cleaned Text)

In [ ]:
cv = CountVectorizer(
    stop_words="english",
    max_features=5000
)

X = cv.fit_transform(sample_df["clean_text"])

top_words = pd.DataFrame({
    "word": cv.get_feature_names_out(),
    "count": X.sum(axis=0).A1
}).sort_values("count", ascending=False)

top_words.head(20)

In [ ]:
fig = px.bar(
    top_words.head(20),
    x="count",
    y="word",
    orientation="h",
    title="Top 20 Words in Complaint Narratives"
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show("browser")

In [ ]:
bigram_cv = CountVectorizer(
    stop_words="english",
    ngram_range=(2, 2),
    max_features=5000
)

X_bigram = bigram_cv.fit_transform(sample_df["clean_text"])

top_bigrams = pd.DataFrame({
    "bigram": bigram_cv.get_feature_names_out(),
    "count": X_bigram.sum(axis=0).A1
}).sort_values("count", ascending=False)

top_bigrams.head(20)

In [ ]:
trigram_cv = CountVectorizer(
    stop_words="english",
    ngram_range=(3, 3),
    max_features=5000
)

X_trigram = trigram_cv.fit_transform(sample_df["clean_text"])

top_trigrams = pd.DataFrame({
    "trigram": trigram_cv.get_feature_names_out(),
    "count": X_trigram.sum(axis=0).A1
}).sort_values("count", ascending=False)

top_trigrams.head(20)

### Word Cloud

Quick visual sanity check on the most frequent terms — same idea as the bar chart above, just a different view.

In [ ]:
from wordcloud import WordCloud

text = " ".join(sample_df["clean_text"])

wordcloud = WordCloud(
    width=1200,
    height=600,
    background_color="white"
).generate(text)

plt.figure(figsize=(15, 8))
plt.imshow(wordcloud)
plt.axis("off")
plt.title("Complaint Narrative Word Cloud")
plt.show()

## 5. Topic Modeling (LDA)

Fit a small LDA model on this notebook's 50,000-row sample to explore latent topics, then map topic IDs to human-readable names for illustration.

**Canonical source:** the production topic labels live in exactly one place, `TOPIC_LABELS` in `src/topic_discovery.py` — imported below rather than redefined here. That mapping describes the production topic model (`models/nlp/topic_model.pkl`, trained on up to 300,000 narratives), which is what `nlp_predictor.py` and the dashboard's Complaint Analyzer tab actually use. Because LDA topic ordering isn't deterministic across separate fits, the exploratory `lda` model trained in this notebook on a smaller sample may assign a different topic ID to the same real-world theme — so the labels below are an illustrative pairing for this notebook's own model, not a guarantee that notebook topic ID *N* means the same thing as production topic ID *N*.

In [ ]:
topic_cv = CountVectorizer(
    stop_words="english",
    max_features=5000,
    min_df=5
)

X_topic = topic_cv.fit_transform(sample_df["clean_text"])

lda = LatentDirichletAllocation(
    n_components=10,
    random_state=42
)

lda.fit(X_topic)

In [ ]:
feature_names = topic_cv.get_feature_names_out()

for topic_idx, topic in enumerate(lda.components_):
    top_words = [
        feature_names[i]
        for i in topic.argsort()[-10:][::-1]
    ]
    print(f"\nTopic {topic_idx}")
    print(top_words)

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

from src.topic_discovery import TOPIC_LABELS

TOPIC_LABELS

In [ ]:
def predict_topic(text: str) -> str:
    cleaned = clean_text(text)
    vector = topic_cv.transform([cleaned])
    topic_probs = lda.transform(vector)
    topic_id = topic_probs.argmax()
    return TOPIC_LABELS.get(topic_id, f"Topic {topic_id}")


sample_text = """
Someone opened fraudulent accounts in my name
and there are inquiries on my credit report.
"""

predict_topic(sample_text)

## 6. Product Classifier

Baseline: TF-IDF + Logistic Regression to predict `Product` from the cleaned narrative. Rare product classes (fewer than 100 samples in this 50K sample) are dropped first since they don't have enough examples to train or evaluate on reliably.

In [ ]:
df_model = sample_df[["clean_text", "Product"]].copy()

product_counts = df_model["Product"].value_counts()
valid_products = product_counts[product_counts >= 100].index

df_model = df_model[df_model["Product"].isin(valid_products)].copy()

df_model.shape

In [ ]:
X = df_model["clean_text"]
y = df_model["Product"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

In [ ]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=10000
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

X_train_tfidf.shape

In [ ]:
model = LogisticRegression(
    max_iter=1000,
    n_jobs=-1
)

model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))

### Tuned Version

Adding bigrams, `class_weight="balanced"` (to help rarer product classes), and a slightly higher `C` for less regularization. This tuned configuration is what `nlp_model_training.py` uses for the full-scale training run.

In [ ]:
tfidf_tuned = TfidfVectorizer(
    stop_words="english",
    max_features=20000,
    ngram_range=(1, 2),
    min_df=3
)

X_train_tfidf_tuned = tfidf_tuned.fit_transform(X_train)
X_test_tfidf_tuned = tfidf_tuned.transform(X_test)

model_tuned = LogisticRegression(
    max_iter=2000,
    n_jobs=-1,
    class_weight="balanced",
    C=2
)

model_tuned.fit(X_train_tfidf_tuned, y_train)

y_pred_tuned = model_tuned.predict(X_test_tfidf_tuned)

print("Accuracy:", accuracy_score(y_test, y_pred_tuned))
print(classification_report(y_test, y_pred_tuned, zero_division=0))

In [ ]:
sample_text = """
Someone opened fraudulent accounts in my name and there are inquiries on my credit report.
"""

sample_text_tfidf = tfidf_tuned.transform([sample_text])
model_tuned.predict(sample_text_tfidf)[0]

## 7. Issue Classifier

Same TF-IDF + Logistic Regression approach, this time predicting `Issue`. Rare issue classes (fewer than 50 samples) are dropped for the same reason as above.

In [ ]:
df_issue = sample_df[["clean_text", "Issue"]].dropna().copy()

issue_counts = df_issue["Issue"].value_counts()
valid_issues = issue_counts[issue_counts >= 50].index

df_issue = df_issue[df_issue["Issue"].isin(valid_issues)].copy()

print(df_issue.shape)
print(df_issue["Issue"].nunique())

In [ ]:
X_issue = df_issue["clean_text"]
y_issue = df_issue["Issue"]

X_train_issue, X_test_issue, y_train_issue, y_test_issue = train_test_split(
    X_issue,
    y_issue,
    test_size=0.2,
    random_state=42,
    stratify=y_issue
)

In [ ]:
tfidf_issue = TfidfVectorizer(
    stop_words="english",
    max_features=10000
)

X_train_issue_tfidf = tfidf_issue.fit_transform(X_train_issue)
X_test_issue_tfidf = tfidf_issue.transform(X_test_issue)

In [ ]:
issue_model = LogisticRegression(
    max_iter=1000,
    n_jobs=-1
)

issue_model.fit(X_train_issue_tfidf, y_train_issue)

y_pred_issue = issue_model.predict(X_test_issue_tfidf)

print("Accuracy:", accuracy_score(y_test_issue, y_pred_issue))
print(classification_report(y_test_issue, y_pred_issue, zero_division=0))

## 8. Summary

- TF-IDF + Logistic Regression gives a reasonable baseline for both Product and Issue prediction on cleaned narrative text.
- The tuned configuration (bigrams, `class_weight="balanced"`, `C=2`) is carried forward into `nlp_model_training.py` for the full-scale training run on up to 300K narratives.
- LDA topic modeling produces 10 latent topics. This notebook imports the canonical `TOPIC_LABELS` mapping from `src/topic_discovery.py` (the single source of truth) rather than defining its own — see the note in Section 5 about why notebook topic IDs and production topic IDs aren't guaranteed to align.
- The simple `predict_topic()` demo above returns only a bare label. The production path (`src/nlp_predictor.py`, used by the dashboard's Complaint Analyzer tab) goes further: Product/Issue predictions expose a canonical-label confidence score, top alternatives, and an AUTO_ACCEPT / HUMAN_REVIEW decision — see `docs/taxonomy_audit.md` and `docs/evaluation.md`.